## Введение

В этом домашнем задании вам предстоит сравнить LLM с разными квантизациями и построить небольшой бенчмарк качества, скорости генерации и потребления памяти.

Для запуска моделей будем использовать [Ollama](https://ollama.com/) — локальный inference server, к которому можно обращаться через API. Ollama поддерживает CPU и совместимые GPU; доступное оборудование влияет на скорость и размещение модели в RAM/VRAM.

Ollama можно запускать **на своём компьютере** или **в Google Colab**. Выберите один из вариантов ниже.


## Запуск Ollama

### Вариант 1. Локально (CPU или GPU)

1. Установите [Ollama](https://ollama.com/download) для вашей ОС.
2. Проверьте, что сервер запущен. Обычно десктопное приложение запускает его автоматически; если нет, выполните `ollama serve` **в отдельном терминале**, оставив процесс работающим.
3. Команды `ollama pull`, `ollama run` и Python-код ниже будут обращаться к этому серверу.

При наличии поддерживаемой GPU Ollama может использовать её автоматически; без совместимой GPU инференс работает на CPU, обычно медленнее. Точное размещение модели можно посмотреть через `ollama ps` после загрузки модели.

[Поддерживаемые GPU](https://docs.ollama.com/gpu).


### Вариант 2. Google Colab (GPU)

1. Выберите **Runtime → Change runtime type → GPU** и убедитесь, что видеокарта доступна (`!nvidia-smi`).
2. Выполните следующие **две Code-ячейки**: установите Ollama, затем запустите сервер в фоне, чтобы он не блокировал ноутбук.
3. После этого можно скачивать модели и обращаться к серверу по `http://localhost:11434`, как при локальном запуске.

*Эти две ячейки нужны только для Colab.* Выбирайте модель и квантизации в соответствии с доступной VRAM. Colab предоставляет временное окружение: после его сброса установку и скачивание моделей может потребоваться повторить.


In [ ]:
# Выполняйте только в Google Colab
!sudo apt-get update -qq
!sudo apt-get install -y -qq pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh
%pip install -q langchain-ollama huggingface-hub


In [ ]:
# Выполняйте только в Google Colab
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434/api/tags"

def ollama_is_ready():
    try:
        return requests.get(OLLAMA_URL, timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_is_ready():
    with open("/tmp/ollama.log", "w") as log:
        ollama_process = subprocess.Popen(
            ["ollama", "serve"],
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    for _ in range(30):
        if ollama_is_ready():
            break
        if ollama_process.poll() is not None:
            raise RuntimeError("Не удалось запустить Ollama. См. /tmp/ollama.log")
        time.sleep(1)
    else:
        raise RuntimeError("Ollama не отвечает. См. /tmp/ollama.log")

print("Ollama API is ready:", OLLAMA_URL)


### Скачивание и запуск моделей (в обоих вариантах)

Когда сервер работает, можно скачать модель из [Ollama Library](https://ollama.com/library) и отправить ей запрос. Например:

```bash
ollama pull qwen2.5:7b-instruct-q4_K_M
ollama run qwen2.5:7b-instruct-q4_K_M "Реши уравнение x^2 = 1"
ollama ps
```

В Jupyter/Colab перед такими командами добавьте `!`, чтобы выполнить их в shell. `ollama run` также скачает модель автоматически, если её ещё нет локально. `ollama ps` показывает загруженные модели и размещение в CPU/GPU; сразу после одного лишь `ollama pull` список может быть пуст.

**Альтернатива:** можно скачать GGUF с Hugging Face и самостоятельно создать Ollama-модель через [Modelfile](https://docs.ollama.com/modelfile). Пример приведён ниже. Этот шаг необязателен, если используете готовые модели из Ollama Library.


In [168]:
!cat Modelname_1.txt

FROM qwen2.5-7b-instruct-q2_k.gguf
PARAMETER temperature 0.7
PARAMETER num_gpu 0


С помощью `huggingface_hub` можно скачать нужные модельки прямо на сервер. Например:

In [14]:
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='Qwen/Qwen2.5-7B-Instruct-GGUF', filename='qwen2.5-7b-instruct-q2_k.gguf', local_dir='.')

'qwen2.5-7b-instruct-q2_k.gguf'

После скачивания GGUF можно зарегистрировать модель через Modelfile (из папки `lecture_2`):

```bash
ollama create qwen2.5-7b-instruct-int2 -f Modelname_1.txt
```

Здесь `Modelname_1.txt` указывает на локальный файл GGUF. Вы можете задать другое имя модели при создании.


1. Можно напрямую отправлять запросы либо использовать python API - https://docs.ollama.com/quickstart
2. Через OpenAI - https://ollama.com/blog/openai-compatibility

Но мы будем использовать удобную обертку из langchain - `ChatOllama` (https://python.langchain.com/docs/integrations/chat/ollama/). Мы подробнее поговорим на следующуем уроке, почему это удобно. Пока просто думаем о `ChatOllama` как о клиенте, который за нас формирует запросы и отправляет из на сервер

In [120]:
from langchain_ollama import ChatOllama

Для построения бенчмарка используем реальные задачи с числовыми ответами из вариантов профильного ЕГЭ за 2025–2026 годы. Условия кратко адаптированы для текстовой модели, а ссылки на исходные задания указаны в датасете.

Мы можем заставить модель выдавать только число на выходе с помощью `structured_output` (https://platform.openai.com/docs/guides/structured-outputs). Для этого можно задать специальную json схему или `pydantic` схему - https://docs.pydantic.dev/latest/ \
С помощью схем мы структурируем выход модели под определенный формат.

## Задания

Подумайте, как должна выглядить схема в случае, если мы хотим, чтобы модель выдала как полноценный ответ на задачу, так и числовой ответ

In [121]:
from typing import List
from pydantic import BaseModel, Field

class MathReasoning(BaseModel):
    pass

Зайдате входной промпт, подумайте, что написать в системном промтпе:

In [166]:
messages = [
 ...   
]

Создадим инстанс модели, указав необходимые параметры

In [133]:
model = 'qwen2.5:7b-instruct-q2_K' # пример: можно выбрать другую модель и квантизацию
ollama_model = ChatOllama(
    model=model,
    api_key="ollama",
    base_url="http://localhost:11434/",
    format=MathReasoning.model_json_schema(), # параметр, отвечающий за выход модели (structured output)
    temperature=0,
)

Используем метод `with_structured_output` из langchain, которые парсит выход модели и валидирует через pydantic схему

In [129]:
# проверьте, что все отрабатывается корректно

output = ollama_model.with_structured_output(schema=MathReasoning).invoke(
    messages,
)

Но если вы прокинули `format` при создании модели, вызвать модель можно и через обычный `invoke`, а затем распарсить и провалидировать выход модели

In [144]:
# проверьте, что все отрабатывается корректно

output = ollama_model.invoke(
    messages,
)
parsed = MathReasoning.model_validate(json.loads(output.content))
print(parsed.final_answer) # достаем ответ как атрибут pydantic схемы

Будем сравнивать модели с разной квантизацией.

Для этого вам надо будет:

1. Выберите LLM и сравните **не менее трёх вариантов квантизации одной и той же модели**. Например, можно использовать Qwen2.5-7B-Instruct с квантизациями `Q2_K`, `Q4_K_M` и `Q8_0`: [доступные варианты в Ollama Library](https://ollama.com/library/qwen2.5/tags). Модели можно скачать командой `ollama pull` или самостоятельно загрузить GGUF через Modelfile.

2. Реализовать расчёт бенчмарка. В результатах нужно сравнить правильность ответов, скорость генерации и потребление RAM/VRAM. Представьте результаты в таблице: model vs accuracy/metrics.

3. Подумайте, какие параметры генерации нужно зафиксировать (temperature и т. д.), чтобы сравнение моделей было честным. Эти параметры необходимо передать при инициализации модели через `ChatOllama`.

Реализуйте расчёт метрик.

**Подсказка:** часть метрик можно получить непосредственно из [Ollama API](https://docs.ollama.com/api/generate), например количество сгенерированных токенов (`eval_count`) и продолжительность генерации (`eval_duration`). Для оценки памяти загруженной модели можно использовать `ollama ps` или [`GET /api/ps`](https://docs.ollama.com/api/ps). Измеряйте память после загрузки модели, пока она остаётся в RAM/VRAM. Подумайте, какие метрики нужно измерить на стороне клиента, в том числе с помощью streaming.

In [ ]:
def measure_inference():
    ...
    return {
        "ttft": ...,           # Time To First Token — время до первого токена (с)
        "ttl": ...,            # Time To Last Token — время до последнего токена (с)
        "completion_tokens": ...,  # Количество сгенерированных токенов
        "overall_time": ...,   # Полное время выполнения запроса (с)
        "output": ...,         # Сгенерированный ответ модели

        # Inter-Token Latency:
        "itl_ms_avg": ...,     # Средний интервал между токенами (мс)

        # Throughput:
        "tokens_per_sec_w/o_prefill": ...,  # Скорость генерации без prefill (токенов/с)
        "tokens_per_sec": ...               # Общая скорость, включая prefill (токенов/с)
    }


def get_memory(model_name: str):
    """
    Оценить объём RAM и VRAM, занимаемый загруженной моделью (MB).
    Пиковое потребление памяти во время генерации измерять необязательно.
    """
    ...
    return {
        "ram_mb": ...,   # Оценка памяти модели в RAM (MB)
        "vram_mb": ...,  # Оценка памяти модели в VRAM (MB)
    }

Реализуйте основную функцию для прогона моделей и расчёта метрик.

Для более устойчивых результатов **желательно повторить бенчмарк несколько раз** (`num_reps`, например 3) и посчитать для каждой модели среднее значение и стандартное отклонение метрик.

Перед измерениями выполните **warm-up**: отправьте хотя бы один пробный запрос каждой модели, чтобы она успела загрузиться. **Не включайте прогревочный запрос в статистику.**

In [ ]:
from typing import List

MODELS = []  # ваш список моделей
BENCH_DATASET = 'ege_math_2025_2026.csv'  # путь к датасету (лежит рядом с ноутбуком)


def compute_benchmark(
    dataset_path: str,
    models: List[str],
    num_reps: int = 3,
    output_savepath: str | None = None,
):
    """
    Прогоните каждую модель по всему датасету num_reps раз (warm-up не учитывать).
    На выходе — таблица агрегированных метрик, при необходимости сохранённая в файл.
    """
    ...

## Ваши выводы

Опишите кратко ваши результаты: как влияет квантизация на качество, скорость генерации; другие инсайты, если есть